# CAPTCHA Recognition Training Notebook

This notebook provides a complete solution for training and evaluating CAPTCHA recognition models.

**Requirements:**
- Dataset directory with CAPTCHA images (default: `samples/`)
- All images should be named with their labels (e.g., `abc12.png` for a CAPTCHA showing "abc12")

**No external files needed** - everything is self-contained in this notebook!


In [1]:
# Install required packages (uncomment if needed)
# !pip install torch torchvision pillow tqdm tensorboard numpy


## 1. Import Libraries


In [2]:
import os
import random
import sys
import platform
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from PIL import Image
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

if platform.system() == 'Darwin':
    import multiprocessing
    multiprocessing.set_start_method('spawn', force=True)


## 2. Model Architectures


In [3]:
class AddNoise:
    """Custom transform to add noise to tensors. Picklable for multiprocessing."""
    def __init__(self, noise_factor=0.05):
        self.noise_factor = noise_factor
    
    def __call__(self, tensor):
        noise = torch.randn_like(tensor) * self.noise_factor
        return torch.clamp(tensor + noise, -1, 1)


class CaptchaDataset(Dataset):
    """Dataset class for CAPTCHA images."""
    
    CHARS = '0123456789abcdefghijklmnopqrstuvwxyz'
    NUM_CHARS = len(CHARS)
    CHAR_TO_IDX = {char: idx for idx, char in enumerate(CHARS)}
    IDX_TO_CHAR = {idx: char for idx, char in enumerate(CHARS)}
    SEQ_LENGTH = 5
    
    def __init__(self, data_dir, image_files=None, transform=None, augment=False):
        self.data_dir = data_dir
        self.transform = transform
        self.augment = augment
        
        if image_files is None:
            self.image_files = []
            for file in os.listdir(data_dir):
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    label = os.path.splitext(file)[0]
                    if len(label) == self.SEQ_LENGTH:
                        self.image_files.append(file)
        else:
            self.image_files = image_files
        
        if self.augment and self.transform is None:
            self.transform = self._get_augmentation_transforms()
        elif self.transform is None:
            self.transform = self._get_base_transforms()
    
    def _get_base_transforms(self):
        return transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((50, 200)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])
    
    def _get_augmentation_transforms(self):
        return transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((50, 200)),
            transforms.RandomRotation(degrees=5),
            transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5]),
            AddNoise(noise_factor=0.05)  # Use picklable class instead of lambda
        ])
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.data_dir, self.image_files[idx])
        image = Image.open(img_path)
        
        if self.transform:
            image = self.transform(image)
        
        label_str = os.path.splitext(self.image_files[idx])[0]
        label = torch.zeros(self.SEQ_LENGTH, dtype=torch.long)
        for i, char in enumerate(label_str):
            if char in self.CHAR_TO_IDX:
                label[i] = self.CHAR_TO_IDX[char]
            else:
                label[i] = 0
        
        return image, label, label_str
    
    @staticmethod
    def decode_label(label_tensor):
        if isinstance(label_tensor, torch.Tensor):
            label_tensor = label_tensor.cpu().numpy()
        return ''.join([CaptchaDataset.IDX_TO_CHAR[int(idx)] for idx in label_tensor])


def create_dataloaders(data_dir, train_split=0.80, batch_size=32, augment_train=True, num_workers=None):
    """Create train and validation dataloaders.
    
    Args:
        num_workers: Number of worker processes. If None, automatically set:
                     - 0 on macOS (for compatibility)
                     - 4 on other systems
    """
    if num_workers is None:
        if platform.system() == 'Darwin': 
            num_workers = 0 
        else:
            num_workers = 4 
    image_files = []
    for file in os.listdir(data_dir):
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            label = os.path.splitext(file)[0]
            if len(label) == CaptchaDataset.SEQ_LENGTH:
                image_files.append(file)
    
    random.seed(42)
    random.shuffle(image_files)
    
    train_size = int(train_split * len(image_files))
    train_files = image_files[:train_size]
    val_files = image_files[train_size:]
    
    train_dataset = CaptchaDataset(data_dir, train_files, augment=augment_train)
    val_dataset = CaptchaDataset(data_dir, val_files, augment=False)
    
    # Disable pin_memory on macOS when using CPU (not needed and can cause issues)
    pin_memory = torch.cuda.is_available() and platform.system() != 'Darwin'
    
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory
    )
    
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory
    )
    
    return train_loader, val_loader


In [4]:
class BaselineCNN(nn.Module):
    """Baseline CNN with 5 separate output heads."""
    
    def __init__(self, num_chars=36, seq_length=5):
        super(BaselineCNN, self).__init__()
        self.num_chars = num_chars
        self.seq_length = seq_length
        # conv layers
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(512)
        self.pool4 = nn.MaxPool2d(2, 2)
        
        self.flatten_size = 512 * 3 * 12
        
        self.fc1 = nn.Linear(self.flatten_size, 1024)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(1024, 512)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(512, 256)
        self.dropout3 = nn.Dropout(0.3)
        
        self.heads = nn.ModuleList([
            nn.Linear(256, num_chars) for _ in range(seq_length)
        ])
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize weights using Kaiming/He initialization for ReLU layers."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool3(x)
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool4(x)
        
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = F.relu(self.fc3(x))
        x = self.dropout3(x)
        
        outputs = [head(x) for head in self.heads]
        return torch.stack(outputs, dim=1)


In [5]:
def train_epoch(model, train_loader, criterion, optimizer, device, max_grad_norm=1.0):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    num_batches = 0
    
    for images, labels, label_strs in tqdm(train_loader, desc='Training'):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = 0
        for i in range(5):
            loss += criterion(outputs[:, i, :], labels[:, i])
        loss = loss / 5
        
        loss.backward()
        
        # Gradient clipping for training stability
        if max_grad_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches


def validate(model, val_loader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss = 0
    num_batches = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels, label_strs in tqdm(val_loader, desc='Validating'):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = 0
            for i in range(5):
                loss += criterion(outputs[:, i, :], labels[:, i])
            loss = loss / 5
            predictions = torch.argmax(outputs, dim=2)
            
            total_loss += loss.item()
            num_batches += 1
            
            for i in range(predictions.size(0)):
                all_predictions.append(predictions[i].cpu().numpy())
                all_labels.append(labels[i].cpu().numpy())
    
    char_acc, seq_acc = calculate_accuracy(all_predictions, all_labels)
    return total_loss / num_batches, char_acc, seq_acc


def calculate_accuracy(predictions, labels):
    """Calculate per-character and full-sequence accuracy."""
    predictions = np.array(predictions)
    labels = np.array(labels)
    
    char_correct = (predictions == labels).sum()
    char_total = predictions.size
    char_acc = char_correct / char_total
    
    seq_correct = (predictions == labels).all(axis=1).sum()
    seq_total = len(predictions)
    seq_acc = seq_correct / seq_total
    
    return char_acc, seq_acc


def evaluate_model(model, dataloader, device):
    """Evaluate a model on a dataset."""
    model.eval()
    all_predictions = []
    all_labels = []
    all_label_strs = []
    
    with torch.no_grad():
        for images, labels, label_strs in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            predictions = torch.argmax(outputs, dim=2)
            
            for i in range(predictions.size(0)):
                all_predictions.append(predictions[i].cpu().numpy())
                all_labels.append(labels[i].cpu().numpy())
                all_label_strs.append(label_strs[i])
    
    char_acc, seq_acc = calculate_accuracy(all_predictions, all_labels)
    pred_strings = [CaptchaDataset.decode_label(pred) for pred in all_predictions]
    
    return char_acc, seq_acc, pred_strings, all_label_strs


def predict_image(model, image_path, device):
    """Predict CAPTCHA from a single image file."""
    model.eval()
    
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((50, 200)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    image = Image.open(image_path)
    image = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(image)
        predictions = torch.argmax(outputs, dim=2)
    
    prediction = CaptchaDataset.decode_label(predictions[0])
    return prediction


In [6]:
# ============================================================================
# CONFIGURATION - Modify these parameters as needed
# ============================================================================

DATA_DIR = 'samples'

MODEL_TYPE = 'baseline'

# training parameters
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 0.0001  # Low and constant learning rate
LABEL_SMOOTHING = 0.1 
USE_AUGMENTATION = True 

# directories for saving models and logs
SAVE_DIR = 'checkpoints'
LOG_DIR = 'logs'

# create directories if they don't exist
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Configuration:")
print(f"  Data directory: {DATA_DIR}")
print(f"  Model type: {MODEL_TYPE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Label smoothing: {LABEL_SMOOTHING}")
print(f"  Augmentation: {USE_AUGMENTATION}")


Configuration:
  Data directory: samples
  Model type: baseline
  Batch size: 32
  Epochs: 30
  Learning rate: 0.0001
  Label smoothing: 0.1
  Augmentation: True


## 5. Load Data


In [7]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load data
print(f'\nLoading data from {DATA_DIR}...')
train_loader, val_loader = create_dataloaders(
    DATA_DIR, 
    train_split=0.8,
    batch_size=BATCH_SIZE, 
    augment_train=USE_AUGMENTATION
)

print(f'Training samples: {len(train_loader.dataset)}')
print(f'Validation samples: {len(val_loader.dataset)}')


Using device: cpu

Loading data from samples...
Training samples: 856
Validation samples: 214


## 6. Create Model


In [8]:
num_chars = CaptchaDataset.NUM_CHARS

# Create model
if MODEL_TYPE == 'baseline':
    model = BaselineCNN(num_chars=num_chars, seq_length=5)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
else:
    model = CRNN(num_chars=num_chars, hidden_size=128, num_layers=2)
    criterion = nn.CTCLoss(blank=num_chars, reduction='mean', zero_infinity=True)

model = model.to(device)

# Create optimizer and scheduler
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

# Use cosine annealing with warm restarts for better convergence
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# TensorBoard writer
writer = SummaryWriter(LOG_DIR)

print(f'Model created: {MODEL_TYPE}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')


Model created: baseline
Total parameters: 21,129,524


## 7. Training Loop


In [9]:
best_seq_acc = 0.0
training_history = {
    'train_loss': [],
    'val_loss': [],
    'char_acc': [],
    'seq_acc': []
}

print(f'\nStarting training ({MODEL_TYPE} model)...')
print('=' * 60)

for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch + 1}/{EPOCHS}')
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, max_grad_norm=1.0)
    
    # Validate
    val_loss, char_acc, seq_acc = validate(model, val_loader, criterion, device)
    
    # Step scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Log to TensorBoard
    writer.add_scalar('Loss/Train', train_loss, epoch)
    writer.add_scalar('Loss/Validation', val_loss, epoch)
    writer.add_scalar('Accuracy/Character', char_acc, epoch)
    writer.add_scalar('Accuracy/Sequence', seq_acc, epoch)
    writer.add_scalar('LearningRate', current_lr, epoch)
    
    # Store history
    training_history['train_loss'].append(train_loss)
    training_history['val_loss'].append(val_loss)
    training_history['char_acc'].append(char_acc)
    training_history['seq_acc'].append(seq_acc)
    
    # Print results
    print(f'Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
    print(f'Character Accuracy: {char_acc:.2%}, Sequence Accuracy: {seq_acc:.2%}')
    print(f'Learning Rate: {current_lr:.6f}')
    
    # Save best model
    should_save = False
    if seq_acc > best_seq_acc:
        best_seq_acc = seq_acc
        should_save = True
    elif seq_acc == 0 and best_seq_acc == 0 and epoch == 0:
        should_save = True
    
    if should_save:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'char_acc': char_acc,
            'seq_acc': seq_acc,
            'model_type': MODEL_TYPE
        }
        best_path = os.path.join(SAVE_DIR, f'best_{MODEL_TYPE}.pth')
        try:
            torch.save(checkpoint, best_path)
            if seq_acc > 0:
                print(f'✓ Saved best model (Sequence Accuracy: {seq_acc:.2%})')
            else:
                print(f'✓ Saved model (Character Accuracy: {char_acc:.2%})')
        except Exception as e:
            print(f'⚠ Warning: Failed to save best model: {e}')
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'char_acc': char_acc,
            'seq_acc': seq_acc,
            'model_type': MODEL_TYPE
        }
        checkpoint_path = os.path.join(SAVE_DIR, f'checkpoint_epoch_{epoch+1}.pth')
        try:
            torch.save(checkpoint, checkpoint_path)
            print(f'✓ Saved checkpoint: {checkpoint_path}')
        except Exception as e:
            print(f'⚠ Warning: Failed to save checkpoint: {e}')

writer.close()
print(f'\n{"=" * 60}')
print(f'Training completed! Best sequence accuracy: {best_seq_acc:.2%}')



Starting training (baseline model)...

Epoch 1/30


Validating: 100%|██████████| 7/7 [00:01<00:00,  5.85it/s]


Train Loss: 57.9210, Val Loss: 7.1041
Character Accuracy: 4.95%, Sequence Accuracy: 0.00%
Learning Rate: 0.000098
✓ Saved model (Character Accuracy: 4.95%)

Epoch 2/30


Validating: 100%|██████████| 7/7 [00:01<00:00,  5.86it/s]


Train Loss: 4.3011, Val Loss: 3.5801
Character Accuracy: 5.14%, Sequence Accuracy: 0.00%
Learning Rate: 0.000091

Epoch 3/30


...

Train Loss: 3.3006, Val Loss: 3.2960
Character Accuracy: 7.76%, Sequence Accuracy: 0.00%
Learning Rate: 0.000003

Epoch 29/30


Validating: 100%|██████████| 7/7 [00:01<00:00,  5.82it/s]


Train Loss: 3.3020, Val Loss: 3.2948
Character Accuracy: 7.76%, Sequence Accuracy: 0.00%
Learning Rate: 0.000002

Epoch 30/30


Validating: 100%|██████████| 7/7 [00:01<00:00,  5.85it/s]


Train Loss: 3.3078, Val Loss: 3.2944
Character Accuracy: 7.76%, Sequence Accuracy: 0.00%
Learning Rate: 0.000100
✓ Saved checkpoint: checkpoints/checkpoint_epoch_30.pth

Training completed! Best sequence accuracy: 0.00%
